# Generalization: Why Training Error Lies


## Introduction

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/07-Generalization.ipynb)

In [ ]:
#| echo: false

%matplotlib inline
%config InlineBackend.figure_format='retina'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display_html, display, Math, HTML;

Every supervised method in this course -- trees, $k$-NN, regression, neural
networks -- is judged by one question: __how well does it predict on data it has
never seen?__

Today we make that question precise, and see why the most natural answer
(pick the model with the lowest training error) is wrong.

## The Supervised Learning Problem

* You are given some example data, which we'll think of abstractly as tuples $\{(\mathbf{x}_i, y_i)\,|\,i = 1,\dots,N\}$.
    * $\mathbf{x}_i$ are the inputs or independent variables. Its components are called __features.__
    * $y_i$ are the outputs or dependent variables (a.k.a. labels, targets or ground truth).

* Your goal is to learn a rule that allows you to predict $y_j$ for some $\mathbf{x}_j$ that is __not__ in the example data you were given.

* The collection $\{(\mathbf{x}_i, y_i)\,|\,i = 1,\dots,N\}$ is called the __training data.__

* The collection $\{(\mathbf{x}_j, y_j)\,|\,j = 1,\dots,M\}$ is called the __test data.__

## The Supervised Contract

What do we have to assume to make this problem tractable?

We assume two things:

1. There is a set of functions that could be used to predict $y_i$ from $\mathbf{x}_i$.
    * This allows us to turn the learning problem into one that searches through this set for the "right" function.
    * However, this set is probably __very__ large!

2. The rule for predicting $y_i$ from $\mathbf{x}_i$ is the same as the rule for predicting $y_j$ from the new item $\mathbf{x}_j$.
    * Speaking probabilistically, we say that $(\mathbf{x}_i, y_i)$ and $(\mathbf{x}_j, y_j)$ are drawn __independently from the same distribution__ (i.i.d.).

Assumption 2 is the contract. Everything we say about generalization today
holds only if the future looks like the past. When it doesn't (a new sensor,
a new year, a new population) no amount of clever modeling saves you.


## A Toy Example: Polynomial Curve Fitting

**Note**

Based on _Pattern Recognition and Machine Learning,_ Christopher Bishop (2006), Section 1.1.



We generate $N = 10$ points $x_i$ equally spaced in $[0, 1]$, and set

$$
y_i = \sin(2\pi x_i) + \epsilon_i,
$$

where $\epsilon_i$ is Gaussian noise. Many data sets are like this: some
component of $y$ depends on $x$, and some component we treat as random --
"noise" -- because it depends on features we cannot see.

We will fit polynomials of order $k$,

$$
f(x, \mathbf{w}) = \sum_{j = 0}^k w_jx^j,
$$

choosing $\mathbf{w}$ to minimize the __least squares__ training error
$E(\mathbf{w}) = \sum_{n=1}^N [f(x_n, \mathbf{w}) - y_n]^2$.
(How to solve for $\mathbf{w}$ is the subject of [Linear Regression](./10-Regression-I-Linear.qmd).)


In [ ]:
#| echo: false
#| fig-align: center
N = 10
x = np.linspace(0, 1, N)
from numpy.random import default_rng
y = np.sin(2 * np.pi * x) + default_rng(2).normal(size = N, scale = 0.20)
cx = np.linspace(0, 1, 1000)
cy = np.sin(2 * np.pi * cx)
plt.figure(figsize = (5, 3.5))
plt.plot(cx, cy, lw = 2, label = r'$\sin(2\pi x)$')
plt.plot(x, y, 'ro', markersize = 8, fillstyle = 'none', label = 'training data')
plt.xlabel('x', size = 16)
plt.ylabel('y', size = 16)
plt.legend(loc = 'best');

## Model Selection

* $\mathbf{w} = (w_0, \dots, w_k)$ are the __parameters__ of the model; least squares finds the $\mathbf{w}^*$ that __minimizes the error on the training data.__

* But what about choosing $k$, the order of the polynomial?

* A cubic ($k = 3$) is a __different model__ from a quadratic ($k = 2$). The problem of choosing $k$ is called __model selection.__

## Model Selection, cont.

Let's look at constant (order 0), linear (order 1), and cubic (order 3) models,
each fit using the least squares criterion:

In [ ]:
#| echo: false
# y = Aw, A is design matrix 1, [1, x^T], [1, x^T, x^T^2], etc, and w-hat = (A^TA)^-1 A^Ty
import warnings
warnings.filterwarnings('ignore')

def design_matrix(x, k):
    N = len(x)
    A = np.ones(N)
    for i in range(1, k+1):
        A = np.column_stack([A, (x.T)**i])
    return A

def fit_poly(x, y, k):
    A = design_matrix(x, k)
    w_hat = np.linalg.inv(A.T @ A) @ A.T @ y
    return w_hat

w_hat_0 = 1/N * np.sum(y)
w_hat_1 = fit_poly(x, y, 1)
w_hat_3 = fit_poly(x, y, 3)

In [ ]:
#| echo: false
fig, axs = plt.subplots(1, 3, sharey = True, figsize = (10, 3.5))
#
cy = 1000 * [w_hat_0]
pred_y = N * [w_hat_0]
axs[0].plot(cx, cy, lw = 2, label = r'$k$ = 0')
axs[0].plot(x, y, 'ro', markersize = 8, fillstyle = 'none')
axs[0].set_xlabel('x', size = 16)
axs[0].set_ylabel('y', size = 16)
axs[0].set_title(r'$k$ = 0, constant' + '\n' + r'$E(\mathbf{w})$ =' + ' {:0.2f}'.format(np.linalg.norm(y - pred_y)))
#
cy = design_matrix(cx, 1) @ w_hat_1
pred_y = design_matrix(x, 1) @ w_hat_1
axs[1].plot(cx, cy, lw = 2, label = r'$k$ = 1')
axs[1].plot(x, y, 'ro', markersize = 8, fillstyle = 'none')
axs[1].set_xlabel('x', size = 16)
axs[1].set_title(r'$k$ = 1, linear' + '\n' + r'$E(\mathbf{w})$ =' + ' {:0.2f}'.format(np.linalg.norm(y - pred_y)))
#
cy = design_matrix(cx, 3) @ w_hat_3
pred_y = design_matrix(x, 3) @ w_hat_3
axs[2].plot(cx, cy, lw = 2, label = r'$k$ = 3')
axs[2].plot(x, y, 'ro', markersize = 8, fillstyle = 'none')
axs[2].set_xlabel('x', size = 16)
axs[2].set_title('$k$ = 3, cubic' + '\n' + r'$E(\mathbf{w})$ =' + ' {:0.2f}'.format(np.linalg.norm(y - pred_y)))
#
fig.tight_layout();

So it looks like a third-order polynomial ($k$ = 3) is a good fit!

How do we know it's good?   Well, the training error $E(\mathbf{w})$ is small.

## Make training error smaller?

Yes, we can, if we increase the order of the polynomial.

We can reduce the error to zero by setting $k = 9$, we get the following polynomial fit to the data:

In [ ]:
#| echo: false
#| fig-align: center
w_hat_9 = fit_poly(x, y, 9)
cy = design_matrix(cx, 9) @ w_hat_9
plt.figure(figsize = (5, 3))
plt.plot(cx, cy, lw = 2, label = r'$k$ = 9')
plt.plot(x, y, 'ro', markersize = 8, fillstyle = 'none')
plt.xlabel('x', size = 16)
plt.ylabel('y', size = 16)
plt.title(r'$k$ = 9' + '\n' + r'$E(\mathbf{w})$ =' + ' {:0.2f}'.format(0));

So ... is the 9th order polynomial a "better" model for this dataset?

## Absolutely not!

**Why?**

* Informally, the model is very "wiggly".  It seems unlikely that the real data generation process is governed by this curve.

* In other words, we don't expect that, if we had __more__ data from the same source, that this model would do a good job of fitting the additional data.

* We want the model to do a good job of predicting on __future__ data.

* This is called the model's __generalization__ ability.

The 9th degree polynomial would seem to have poor generalization ability.

## Generalization Error



* To assess generalization, we evaluate each polynomial on new __test__ data -- not part of the training set. (We know how the data is generated, so we can easily make more.)

* As we increase the order of the polynomial, the _training error_ always declines.

* Eventually, the training error reaches zero.

* However, the _test error_ does not -- it reaches its smallest value at $k = 3$, a cubic polynomial.

* The phenomenon in which _training error_ declines, but _testing error_ does not, is called __overfitting.__

* In a sense we are fitting the training data "too well".


In [ ]:
#| echo: false
#| fig-align: center
test_y = np.sin(2 * np.pi * x) + default_rng(8).normal(size = N, scale = 0.20)
max_k = N
train_err = [np.linalg.norm(y - N * [w_hat_0])]
test_err = [np.linalg.norm(test_y - N * [w_hat_0])]
for k in range(1, max_k):
    w_hat = fit_poly(x, y, k)
    pred_y = design_matrix(x, k) @ w_hat
    train_err.append(np.linalg.norm(y - pred_y))
    test_err.append(np.linalg.norm(test_y - pred_y))
plt.figure(figsize = (6, 4))
plt.plot(range(max_k), test_err, 'ro-', label = 'Testing Error')
plt.plot(range(max_k), train_err, 'bo-', label = 'Training Error')
plt.xlabel(r'$k$', size = 16)
plt.ylabel(r'$E(\mathbf{w}^*)$')
plt.legend(loc = 'best');

## Overfitting

There are two ways to think about overfitting:

1. The number of parameters in the model is too large, compared to the size of the training data.   We can see this in the fact that we have only 10 training points, and the 9th order polynomial has 10 coefficents.

2. The model is more complex than the actual phenomenon being modeled.  As a result, the model is not just fitting the underlying phenomenon, but also the noise in the data.

These suggest techniques we may use to avoid overfitting:

1. Increase the amount of training data.  All else being equal, more training data is always better.

2. Limit the complexity of the model.  Model complexity is often controlled via __hyperparameters__.

3. Use regularization -- constrain the model to avoid overfitting.

## Bias and Variance

Overfitting is one of __two ways a model can be wrong.__

__Bias__

* Error due to an overly simplistic model.
* High bias: the model __underfits__ -- it misses the real pattern. (Our $k = 0$ or $k = 1$ polynomial.)
* Low bias: the model is flexible enough to capture the underlying pattern.

__Variance__

* Error due to an overly complex model.
* High variance: the model __overfits__ -- refit it on a different training sample from the same distribution and you get a very different answer. (Our $k = 9$ polynomial.)
* Low variance: predictions are stable across different training sets.

---

![](figs/bias_variance_tradeoff.png)

## Bias-Variance Trade-Off

Goal: find the model complexity that minimizes __total__ error.

* Low bias and low variance are both ideal, but hard to achieve simultaneously: making a model more flexible lowers bias and raises variance.

* The U-shaped test error curve we just saw is this trade-off in action -- $k = 3$ is the sweet spot.

![[Source](https://serokell.io/blog/bias-variance-tradeoff)](figs/bias_variance_tradeoff2.png)

## Parameters and Hyperparameters

* Notice that the model selection problem required us to choose a value $k$ that specifies the order of the polynomial model.

* The values $w_0, w_1, \dots, w_k$ are the __parameters__ of the model; they are learned from the training data.

* In contrast, $k$ is called a __hyperparameter.__


* A hyperparameter is a parameter that must be set first, before the (regular) parameters can be learned.

* Hyperparameters are often used to control model complexity.

* So, to avoid overfitting, we need to choose the proper value for the hyperparameter $k$.

* We do that by __holding out data.__


## Holding Out Data

* We want to avoid overfitting, which occurs when a model fails to generalize -- that is, when it has high error on data that it was not trained on.

* So: we will hold some data aside, and __not__ use it for training the model, but instead use it for testing generalization ability.

* Let's assume that we have 20 data points to work with. `scikit-learn`'s `train_test_split()` splits them __randomly__ into training and testing sets:

In [ ]:
#| echo: true
#| code-fold: false
N = 20
x = np.linspace(0, 1, N)
y = np.sin(2 * np.pi * x) + default_rng(2).normal(size = N, scale = 0.20)

import sklearn.model_selection as model_selection

x_train, x_test, y_train, y_test = model_selection.train_test_split(
        x, y, test_size = 0.5, random_state = 0)

print(f'Number of items in training set: {x_train.shape[0]}, in testing set: {x_test.shape[0]}')

## Grid Search

Our strategy will be, for each possible value of the hyperparameter $k$:

* randomly split the data 5 times
* fit the model on the training data
* test the model on the testing data
* compute the mean testing and training error over the 5 random splits

Trying all candidate values of the hyperparameter this way is called a __grid search__.
(What are good candidate values?  It depends on the problem, and may involve trial and error.)

In [ ]:
#| echo: false
def model_error(x_train, y_train, x_test, y_test, k):
    '''
    This function fits a polynomial of degree k to the training data
    and returns the error on both the training and test data.
    '''
    w_star = fit_poly(x_train, y_train, k)
    pred_test_y = design_matrix(x_test, k) @ w_star
    pred_train_y = design_matrix(x_train, k) @ w_star
    return (np.linalg.norm(y_train - pred_train_y), np.linalg.norm(y_test - pred_test_y))

np.random.seed(7)
max_k = 10
n_splits = 5
err = []
for k in range(1, max_k):
    for s in range(n_splits):
        x_train, x_test, y_train, y_test = model_selection.train_test_split(
            x, y, test_size = 0.5)
        split_train_err, split_test_err = model_error(x_train, y_train, x_test, y_test, k)
        err.append([k, s, split_train_err, split_test_err])
df = pd.DataFrame(err, columns = ['k', 'split', 'Training Error', 'Testing Error'])

## Grid Search, cont.

Mean error for each value of `k`, with its standard error ($\sigma/\sqrt{n}$) over the 5 splits:

In [ ]:
#| echo: false
#| fig-align: center
df.groupby('k').mean()[['Training Error', 'Testing Error']].plot(
    yerr = df.groupby('k').std()/np.sqrt(n_splits), figsize = (6, 4))
plt.ylabel('Error')
plt.ylim([0, 5]);

From this plot we can conclude that, for this dataset, a polynomial of degree $k = 3$ shows the best generalization ability.

## Hold Out Strategies

* Deciding how much, and which, data to hold out depends on a number of factors.

* In general we'd like to give the training stage as much data as possible to work with.

* However, the more data we use for training, the less we have for testing -- which can decrease the accuracy of the testing stage.

* Furthermore, any single partition of the data can introduce dependencies -- any class that is overrepresented in the training data will be underrepresented in the test data.


There are two ways to address these problems:

* __Random subsampling__: partition the data randomly between train and test sets, and repeat a reasonable number of times (usually five at least). This is what we just did with `train_test_split()`.
* __$K$-Fold Cross-validation__


## $K$-Fold Cross-Validation

In __$K$-Fold Cross-validation__, the data is partitioned _once_, and then each partition is used as the test data once.

This ensures that all the data gets equal weight in the training and in the testing.



* We divide the data into $k$ "folds".

* The value of $k$ can vary up to the size of the dataset.

* The larger $k$ we use, the more data is used for training, but the more folds must be evaluated, which increases the time required.

* In the extreme case where $k$ is equal to the data size, then each data item is held out by itself; this is called "leave-one-out".



![](figs/L13-k-fold.png)



## How This Plays Out This Term

Every supervised lecture returns to today's ideas -- the same trade-off, a different knob:

* [Decision Trees](./08-Classification-I-Decision-Trees.qmd): overfit via __depth__; random forests reduce variance by averaging.
* [$k$-NN](./09-Classification-II-kNN.qmd): choosing $k$ -- small $k$ overfits, large $k$ underfits.
* [Regression](./11-Regression-II-Logistic-Regularization.qmd): __regularization__ penalizes complexity directly.
* [Neural Networks](./16-Neural-Networks.qmd): __early stopping__ -- halt training when held-out error starts to rise.

In every case, the hyperparameter is chosen by cross-validation, never by training error.

## Conclusions

We have seen strategies that allow us to learn from data:

* Define a set of possible models
* Define an error function that tells us when the model is predicting well
* Using the error function, search through the space of models to find the best performer

We've also seen that there are some subtleties to this approach that must be dealt with to avoid problems:

* Simply using the model that has lowest error on the training data will __overfit__.
* Overfitting (variance) and underfitting (bias) are the two ways a model can be wrong; the goal is to balance them.
* We need to __hold out__ data to assess the generalization ability of each trained model.
* We control model complexity using hyperparameters.
* We choose the best hyperparameters based on performance on held out data.